# Gradient-descent reconstructor

`GDReconstructor` needs no training: for each WFS frame it fits, by gradient descent through the frozen twin, the OPD that makes the twin reproduce that frame. It works on raw WFS frames and returns DM coefficients, so it can be used wherever a reconstructor is evaluated (`Trainer.evaluate`, `evaluate_many`).

This notebook uses the demo instrument of `basics/` (unmodulated Pyramid, `../../basics/wfs_params_exp.py`). Design and prototype results: `Ideas/24-gradient-descent-reconstructor.md`.

In [ ]:
import matplotlib.pyplot as plt
import torch
from mmengine import Config

from AI4AO import (PyramidWFS, DeformableMirror, PhaseDataset, FramePreprocess, Trainer,
                   GDReconstructor, imshow_multiple)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

p = Config.fromfile('../../basics/wfs_params_exp.py')
WFSParams, AtmosParams, LoopParams, DMParams = p.WFSParams, p.AtmosParams, p.LoopParams, p.DMParams

wfs = PyramidWFS(dict(WFSParams, useNoise=False), device).eval()   # noise-free frames for now
dm = DeformableMirror(WFSParams, DMParams, device).eval()
M2C = dm.MakeZernikeM2C()                                          # (totalAct, 64)

dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)   # open-loop screens

## Mode scale

The fit runs on `u` with coefficients `c = mode_scale * u`. `mode_scale` is the typical amplitude of each mode in turbulence (per nm RMS of OPD), measured once on simulated screens. Without it, Adam moves every coefficient by the same step and gets stuck on large aberrations.

In [ ]:
torch.manual_seed(0)
mode_scale = GDReconstructor.mode_scale_from_dataset(dataset, dm, M2C)

plt.semilogy(mode_scale.cpu())
plt.xlabel('mode'); plt.ylabel('typical coefficient per nm RMS')
plt.show()

## One frame, a moderate and a large aberration

The true OPD is a turbulent screen projected onto the DM modes, scaled to 30 and 200 nm RMS. The linear reconstructor (interaction matrix) saturates at 200 nm; gradient descent does not.

In [ ]:
pupil = wfs.pupil.bool()

def rms_nm(opd):
    v = opd[:, pupil]
    return ((v - v.mean(dim=-1, keepdim=True)) ** 2).mean(dim=-1).sqrt() * 1e9

with torch.no_grad():
    modes = dm.GetDMShape(M2C.T)
    wfs.BuildReconstructionMatrix(modes)                     # the linear reconstructor
    z_inv = torch.linalg.pinv(modes.flatten(start_dim=-2))
    torch.manual_seed(1)
    c_unit = dataset[0]['opd'][:1].flatten(start_dim=-2) @ z_inv
    c_unit = c_unit / rms_nm(dm.GetDMShape(c_unit @ M2C.T))[:, None]   # 1 nm RMS

gd = GDReconstructor(wfs, dm, M2C, mode_scale=mode_scale)    # opd_mode='dm', 300 iterations

for amplitude in (30, 200):
    c_true = amplitude * c_unit
    with torch.no_grad():
        opd_true = dm.GetDMShape(c_true @ M2C.T)
        frame = wfs(opd_true)
        opd_linear = dm.GetDMShape(wfs.GetReconstructedOPD(frame) @ M2C.T)
    c_gd = gd(frame)
    print(f"{amplitude} nm: linear error {rms_nm(opd_linear - opd_true).item():6.1f} nm, "
          f"gradient descent {rms_nm(gd.opd - opd_true).item():6.1f} nm")

imshow_multiple([
    {"tensor": opd_true * pupil, "title": "true OPD (200 nm)"},
    {"tensor": opd_linear * pupil, "title": "linear", "scale_reference": opd_true * pupil},
    {"tensor": gd.opd * pupil, "title": "gradient descent", "scale_reference": opd_true * pupil},
    {"tensor": (opd_true - gd.opd) * pupil, "title": "GD error", "scale_reference": opd_true * pupil},
])
plt.show()

## OPD representation: `"dm"`, `"pixel"` or `"dm+pixel"`

As for `TwinCalibrator`'s static OPD: DM commands only, a free per-pixel map, or both (the pixel part learning `residual_lr_factor` times slower). The call always returns DM coefficients; the maps are in `gd.opd`, `gd.dm_opd` and `gd.pixel_opd`. `"pixel"` has no mode scale, so it is exact for small aberrations but can get stuck on large ones.

In [ ]:
for opd_mode in ("dm", "pixel", "dm+pixel"):
    g = GDReconstructor(wfs, dm, M2C, opd_mode=opd_mode, mode_scale=mode_scale)
    g(frame)
    print(f"{opd_mode:9s}: error {rms_nm(g.opd - opd_true).item():6.1f} nm, final loss {g.loss.item():.1e}")

imshow_multiple([
    {"tensor": g.dm_opd * pupil, "title": "dm+pixel: DM part"},
    {"tensor": g.pixel_opd * pupil, "title": "dm+pixel: pixel part"},
], colorbar=True)
plt.show()

## In closed loop

`GDReconstructor` is called like `Trainer.linear_reconstructor()`, on raw (noisy) WFS frames, so `evaluate_many` compares it with the linear reconstructor and the oracle under identical conditions. Each loop step runs `n_iter` iterations for the whole batch, so this is slow compared with a CNN; a few steps on a small batch are enough to see it.

In [ ]:
wfs.useNoise = True
loop_dataset = PhaseDataset(WFSParams, dict(AtmosParams, Nphases=4), LoopParams, DMParams, device)
loop_dataset.generateClosedLoop = True
framePreprocessor = FramePreprocess(WFSParams, wfs, device)
framePreprocessor.ProcessReference(wfs.reference_intensity)
trainer = Trainer(wfs, framePreprocessor, dm, M2C, dataset=loop_dataset)

gd_loop = GDReconstructor(wfs, dm, M2C, mode_scale=mode_scale, n_iter=100)
report = trainer.evaluate_many({"gradient descent": gd_loop}, seeds=(1234,), n_steps=30,
                               references=("linear", "oracle"), baseline="linear")
print(report.table())